# Road Damage Detection — 05. Streamlit Deployment

This notebook generates a polished image-based Streamlit application around the production model
selected in notebook 04. It includes upload and camera input, confidence and resolution controls,
annotated output, counts, latency/FPS, a detection table, downloads, model caching, responsive
styling, and clear empty/error states.

In [ ]:
%pip install -q \
    ultralytics==8.4.92 \
    streamlit>=1.40,<2 \
    pandas>=2.2,<3 \
    pillow>=11,<13 \
    opencv-python-headless>=4.10,<5

## Deployment handoff

This notebook does not need the training dataset. It consumes the production checkpoint
created by notebook 04 at:

```text
/content/drive/MyDrive/RoadDamageYOLO/exports/production_road_damage_model.pt
```

Run notebook 04 successfully before this notebook.

In [ ]:
from __future__ import annotations
import py_compile
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/RoadDamageYOLO')
EXPORTS_ROOT = PROJECT_ROOT / 'exports'
APP_ROOT = PROJECT_ROOT / 'streamlit_app'
APP_ROOT.mkdir(parents=True, exist_ok=True)
PRODUCTION_MODEL = EXPORTS_ROOT / 'production_road_damage_model.pt'
if not PRODUCTION_MODEL.exists():
    raise FileNotFoundError('Run notebook 04 and create the production model first.')
print('App directory:', APP_ROOT)
print('Production model:', PRODUCTION_MODEL)

## 1. Frontend design brief

In [ ]:
design_brief = """# Road Damage Detection — Frontend Design Brief

## Users
Road-maintenance analysts, engineering students, municipal teams, and project reviewers.

## Main journey
Upload or capture an image, adjust settings, run detection, review visual evidence and metrics,
then download the annotated image or structured detections.

## Visual direction
A professional road-infrastructure dashboard: neutral light surfaces, strong dark text, a restrained
safety-yellow accent, clear input-to-result hierarchy, minimal decoration, and the annotated image as
the dominant output.

## Accessibility and responsive behavior
Strong contrast, explicit labels, textual status messages, native keyboard-accessible controls, and
a two-column layout that stacks on narrow screens.

## States
Clear empty, loading, success, no-detection, and error states without claiming engineering certainty.
"""
(APP_ROOT / 'FRONTEND_DESIGN.md').write_text(design_brief, encoding='utf-8')
print(design_brief)

## 2. Generate the application

In [ ]:
app_code = 'from __future__ import annotations\n\nimport os\nimport time\nfrom collections import Counter\nfrom io import BytesIO\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\nfrom PIL import Image, ImageOps, UnidentifiedImageError\nfrom ultralytics import YOLO\n\nAPP_DIR = Path(__file__).resolve().parent\nDEFAULT_MODEL_PATH = APP_DIR / "models" / "best.pt"\nMODEL_PATH = Path(os.getenv("MODEL_PATH", str(DEFAULT_MODEL_PATH)))\nMAX_UPLOAD_MB = 15\nMAX_IMAGE_DIMENSION = 4096\nSUPPORTED_TYPES = ["jpg", "jpeg", "png", "webp"]\n\nst.set_page_config(\n    page_title="Road Damage AI",\n    page_icon="🛣️",\n    layout="wide",\n    initial_sidebar_state="expanded",\n)\n\nst.markdown(\n    """\n    <style>\n    :root {\n        --ink: #172026; --muted: #59656d; --surface: #ffffff;\n        --canvas: #f4f5f2; --line: #dfe3df; --accent: #f4bf28;\n        --accent-dark: #2d2a1f;\n    }\n    .stApp {\n        background:\n            linear-gradient(90deg, rgba(244,191,40,.06) 1px, transparent 1px),\n            linear-gradient(rgba(23,32,38,.035) 1px, transparent 1px),\n            var(--canvas);\n        background-size: 32px 32px; color: var(--ink);\n    }\n    [data-testid="stHeader"] { background: rgba(244,245,242,.9); }\n    .hero {\n        background: var(--ink); border-radius: 22px; padding: 30px 34px;\n        margin-bottom: 22px; position: relative; overflow: hidden;\n    }\n    .hero::after {\n        content: ""; position: absolute; right: -55px; top: -75px;\n        width: 240px; height: 240px; border: 28px solid rgba(244,191,40,.25);\n        border-radius: 50%;\n    }\n    .eyebrow { color: var(--accent); font-size: .82rem; font-weight: 800;\n        letter-spacing: .14em; text-transform: uppercase; }\n    .hero h1 { color: white; margin: 8px 0 10px; font-size: clamp(2rem,4vw,3.35rem);\n        line-height: 1.02; max-width: 850px; }\n    .hero p { color: #d6dde1; margin: 0; max-width: 760px; font-size: 1.02rem; }\n    .section-label { font-size: .78rem; color: var(--muted); font-weight: 800;\n        letter-spacing: .12em; text-transform: uppercase; margin-bottom: 4px; }\n    [data-testid="stMetric"] { background: var(--surface); border: 1px solid var(--line);\n        border-radius: 16px; padding: 14px 16px; box-shadow: 0 8px 24px rgba(23,32,38,.045); }\n    [data-testid="stFileUploader"] { background: var(--surface); border: 1px solid var(--line);\n        border-radius: 16px; padding: 8px; }\n    .status-card { padding: 16px 18px; border: 1px solid var(--line);\n        border-left: 5px solid var(--accent); border-radius: 14px;\n        background: var(--surface); margin: 10px 0 18px; }\n    .stButton > button { border-radius: 12px; border: 1px solid var(--accent-dark);\n        background: var(--accent); color: var(--accent-dark); font-weight: 800; min-height: 46px; }\n    .stButton > button:hover { border-color: var(--ink); background: #ffd45c; color: var(--ink); }\n    [data-testid="stSidebar"] { border-right: 1px solid var(--line); }\n    @media (max-width: 800px) { .hero { padding: 24px 22px; } }\n    </style>\n    """,\n    unsafe_allow_html=True,\n)\n\n\n@st.cache_resource(show_spinner="Loading the road-damage model...")\ndef load_model(model_path: str) -> YOLO:\n    path = Path(model_path)\n    if not path.exists():\n        raise FileNotFoundError(\n            f"Model not found at {path}. Place the checkpoint at models/best.pt."\n        )\n    return YOLO(str(path))\n\n\ndef normalize_image(image: Image.Image) -> Image.Image:\n    image = ImageOps.exif_transpose(image).convert("RGB")\n    if max(image.size) > MAX_IMAGE_DIMENSION:\n        image.thumbnail((MAX_IMAGE_DIMENSION, MAX_IMAGE_DIMENSION))\n    return image\n\n\ndef png_bytes(image: Image.Image) -> bytes:\n    buffer = BytesIO()\n    image.save(buffer, format="PNG")\n    return buffer.getvalue()\n\n\ndef detections_frame(result, names: dict[int, str]) -> pd.DataFrame:\n    columns = ["class", "confidence", "x_min", "y_min", "x_max", "y_max"]\n    if result.boxes is None or len(result.boxes) == 0:\n        return pd.DataFrame(columns=columns)\n    boxes = result.boxes.xyxy.detach().cpu().numpy()\n    confidence = result.boxes.conf.detach().cpu().numpy()\n    classes = result.boxes.cls.detach().cpu().numpy().astype(int)\n    rows = []\n    for box, score, class_id in zip(boxes, confidence, classes):\n        rows.append(\n            {\n                "class": names.get(class_id, str(class_id)),\n                "confidence": float(score),\n                "x_min": float(box[0]),\n                "y_min": float(box[1]),\n                "x_max": float(box[2]),\n                "y_max": float(box[3]),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("confidence", ascending=False).reset_index(drop=True)\n\n\nst.markdown(\n    """\n<section class="hero">\n  <div class="eyebrow">Computer Vision · Road Infrastructure</div>\n  <h1>Find road damage before it becomes a larger failure.</h1>\n  <p>Upload or capture a road image. The selected YOLO detector will localize visible damage and return reviewable evidence in seconds.</p>\n</section>\n""",\n    unsafe_allow_html=True,\n)\n\nwith st.sidebar:\n    st.markdown("## Detection settings")\n    confidence = st.slider(\n        "Confidence threshold",\n        0.05,\n        0.90,\n        0.25,\n        0.05,\n        help="Lower values return more candidates but may increase false positives.",\n    )\n    image_size = st.select_slider(\n        "Inference resolution",\n        options=[480, 640, 800, 960],\n        value=640,\n        help="Higher values may help small damage but increase latency.",\n    )\n    st.divider()\n    st.markdown("### Deployment")\n    st.caption(f"Model: `{MODEL_PATH.name}`")\n    st.caption(\n        "Uploaded images are processed during this session and are not intentionally persisted."\n    )\n    st.divider()\n    st.markdown("### Interpretation note")\n    st.caption(\n        "This tool supports inspection and prioritization. It does not replace an on-site engineering assessment."\n    )\n\ntry:\n    model = load_model(str(MODEL_PATH))\nexcept Exception as exc:\n    st.error(f"Model could not be loaded: {exc}")\n    st.stop()\n\nclass_names = model.names if isinstance(model.names, dict) else dict(enumerate(model.names))\nst.markdown(\'<div class="section-label">01 · Input</div>\', unsafe_allow_html=True)\nst.subheader("Choose a road image")\nupload_tab, camera_tab = st.tabs(["Upload image", "Use camera"])\ninput_file = None\nwith upload_tab:\n    input_file = st.file_uploader(\n        "Drop an image here",\n        type=SUPPORTED_TYPES,\n        help=f"Accepted formats: {\', \'.join(SUPPORTED_TYPES)}. Maximum {MAX_UPLOAD_MB} MB.",\n    )\nwith camera_tab:\n    camera_file = st.camera_input("Capture a road image")\n    if camera_file is not None:\n        input_file = camera_file\n\nif input_file is None:\n    st.markdown(\n        \'<div class="status-card"><strong>Ready for an image.</strong><br>Upload a clear road-surface photo or use the camera tab.</div>\',\n        unsafe_allow_html=True,\n    )\n    st.stop()\n\ntry:\n    file_size_mb = len(input_file.getvalue()) / (1024**2)\n    if file_size_mb > MAX_UPLOAD_MB:\n        raise ValueError(\n            f"The image is {file_size_mb:.1f} MB; maximum is {MAX_UPLOAD_MB} MB."\n        )\n    source_image = normalize_image(Image.open(input_file))\nexcept (UnidentifiedImageError, OSError, ValueError) as exc:\n    st.error(f"The selected file could not be processed: {exc}")\n    st.stop()\n\nleft, right = st.columns([1, 1], gap="large")\nwith left:\n    st.image(\n        source_image,\n        caption=f"Original · {source_image.width} × {source_image.height}px",\n        use_container_width=True,\n    )\n\nrun = st.button("Run road-damage detection", type="primary", use_container_width=True)\nif not run:\n    with right:\n        st.info("Review the image, then run detection.")\n    st.stop()\n\nwith st.spinner("Inspecting the road surface..."):\n    started = time.perf_counter()\n    result = model.predict(\n        source=np.asarray(source_image),\n        conf=confidence,\n        imgsz=image_size,\n        verbose=False,\n    )[0]\n    latency_ms = (time.perf_counter() - started) * 1000\n\ndetections = detections_frame(result, class_names)\nannotated_image = Image.fromarray(\n    result.plot(line_width=2, labels=True, conf=True)[..., ::-1]\n)\nwith right:\n    st.image(annotated_image, caption="Detected road damage", use_container_width=True)\n\nst.markdown(\n    \'<div class="section-label">02 · Result summary</div>\', unsafe_allow_html=True\n)\nst.subheader("Detection summary")\nmetric_1, metric_2, metric_3, metric_4 = st.columns(4)\ncount = len(detections)\nunique = int(detections["class"].nunique()) if count else 0\ntop = float(detections["confidence"].max()) if count else 0.0\nfps = 1000.0 / latency_ms if latency_ms > 0 else 0.0\nmetric_1.metric("Detections", count)\nmetric_2.metric("Damage types", unique)\nmetric_3.metric("Top confidence", f"{top:.1%}")\nmetric_4.metric("End-to-end speed", f"{latency_ms:.1f} ms", f"{fps:.1f} FPS")\n\nif detections.empty:\n    st.warning(\n        "No road damage passed the current threshold. This does not guarantee that the road is damage-free."\n    )\nelse:\n    counts = Counter(detections["class"])\n    text = " · ".join(f"{name}: {value}" for name, value in counts.most_common())\n    st.markdown(\n        f\'<div class="status-card"><strong>Detected classes</strong><br>{text}</div>\',\n        unsafe_allow_html=True,\n    )\n    st.markdown(\n        \'<div class="section-label">03 · Detection evidence</div>\',\n        unsafe_allow_html=True,\n    )\n    st.subheader("Bounding-box details")\n    shown = detections.copy()\n    shown["confidence"] = shown["confidence"].map(lambda value: f"{value:.1%}")\n    for column in ["x_min", "y_min", "x_max", "y_max"]:\n        shown[column] = shown[column].round(1)\n    st.dataframe(shown, use_container_width=True, hide_index=True)\n\nst.markdown(\'<div class="section-label">04 · Export</div>\', unsafe_allow_html=True)\nst.subheader("Download the analysis")\nbutton_1, button_2 = st.columns(2)\nwith button_1:\n    st.download_button(\n        "Download annotated image",\n        png_bytes(annotated_image),\n        "road_damage_detection.png",\n        "image/png",\n        use_container_width=True,\n    )\nwith button_2:\n    st.download_button(\n        "Download detection CSV",\n        detections.to_csv(index=False).encode("utf-8"),\n        "road_damage_detections.csv",\n        "text/csv",\n        use_container_width=True,\n    )\n\nwith st.expander("Runtime details"):\n    st.json(\n        {\n            "model": MODEL_PATH.name,\n            "confidence_threshold": confidence,\n            "inference_resolution": image_size,\n            "wall_latency_ms": round(latency_ms, 3),\n            "ultralytics_speed_ms": {\n                key: round(float(value), 3) for key, value in result.speed.items()\n            },\n            "input_width": source_image.width,\n            "input_height": source_image.height,\n        }\n    )\n'
(APP_ROOT / 'app.py').write_text(app_code, encoding='utf-8')
print('Created:', APP_ROOT / 'app.py')

## 3. Add deployment files and the model

In [ ]:
requirements = """ultralytics==8.4.92
streamlit>=1.40,<2
pandas>=2.2,<3
pillow>=11,<13
opencv-python-headless>=4.10,<5
"""
config = """[theme]
base = "light"
primaryColor = "#F4BF28"
backgroundColor = "#F4F5F2"
secondaryBackgroundColor = "#FFFFFF"
textColor = "#172026"
font = "sans serif"

[server]
maxUploadSize = 15
headless = true
"""
readme = """# Road Damage AI — Streamlit App

Run locally:

```bash
python -m venv .venv
# Windows: .venv\\Scripts\\activate
# Linux/macOS: source .venv/bin/activate
pip install -r requirements.txt
streamlit run app.py
```

The model must be at `models/best.pt`, or set `MODEL_PATH`. For Community Cloud, push this
folder to GitHub, point the app to `app.py`, verify model licensing, and never commit API keys.
"""
(APP_ROOT / 'requirements.txt').write_text(requirements, encoding='utf-8')
(APP_ROOT / 'README.md').write_text(readme, encoding='utf-8')
config_dir = APP_ROOT / '.streamlit'; config_dir.mkdir(exist_ok=True)
(config_dir / 'config.toml').write_text(config, encoding='utf-8')
models_dir = APP_ROOT / 'models'; models_dir.mkdir(exist_ok=True)
shutil.copy2(PRODUCTION_MODEL, models_dir / 'best.pt')
for path in sorted(APP_ROOT.rglob('*')):
    if path.is_file():
        print(path.relative_to(APP_ROOT), f'({path.stat().st_size / 1024:.1f} KB)')

## 4. Syntax and model-loading smoke tests

In [ ]:
from ultralytics import YOLO
py_compile.compile(str(APP_ROOT / 'app.py'), doraise=True)
print('app.py syntax: OK')
model = YOLO(str(APP_ROOT / 'models' / 'best.pt'))
print('Model load: OK')
print('Classes:', model.names)

## 5. Package the app

In [ ]:
archive = shutil.make_archive(
    str(PROJECT_ROOT / 'road_damage_streamlit_app'),
    'zip', root_dir=APP_ROOT,
)
print('Created:', archive)

## Completed

App folder: `MyDrive/RoadDamageYOLO/streamlit_app`  
ZIP: `MyDrive/RoadDamageYOLO/road_damage_streamlit_app.zip`